# Taiwan PyPSA-Earth Simulation Results

This notebook is a first-pass results dashboard for the completed Taiwan run.

PyPSA-Earth's local checkout does not include a result-analysis notebook, but the tutorial docs point to the external documentation example `sample_network_analysis.ipynb` in `pypsa-meets-earth/documentation`.

It loads the solved network from:

`../results/networks/elec_s_6_ec_lcopt_Co2L-4H.nc`

## What To Show

- Run summary: objective value, snapshots, buses, lines, generators, stores, and storage units.
- Capacity mix: installed/optimized generator capacity by carrier.
- Energy mix: total generated electricity by carrier as an interactive Plotly pie chart.
- Demand and dispatch: interactive time series showing total load and carrier-level dispatch.
- Renewable curtailment: estimated available renewable energy minus dispatched renewable energy.
- Spatial overview: interactive Plotly map with OpenStreetMap basemap, Taiwan buses, lines, and generator capacity bubbles.
- Sanity checks: load served, missing values, suspicious negative dispatch, and solved output path.

In [ ]:
from pathlib import Path

import pandas as pd

try:
    import plotly.express as px
    import plotly.graph_objects as go
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "This notebook uses Plotly. Install it in the pypsa-earth environment with: "
        "python -m pip install plotly"
    ) from exc

import pypsa

pd.options.display.float_format = "{:,.2f}".format

RESULT_PATH = Path("../results/networks/elec_s_6_ec_lcopt_Co2L-4H.nc")
RESULT_PATH

In [ ]:
if not RESULT_PATH.exists():
    raise FileNotFoundError(
        f"Solved network not found: {RESULT_PATH}. Run `snakemake -j 1 solve_all_networks` first."
    )

n = pypsa.Network(RESULT_PATH)
n

## Run Summary

This is the quick health check: did we load the solved file, how large is it, and what objective value did the optimization report?

In [ ]:
summary = pd.Series(
    {
        "result_file": str(RESULT_PATH),
        "objective": getattr(n, "objective", None),
        "snapshots": len(n.snapshots),
        "first_snapshot": n.snapshots[0] if len(n.snapshots) else None,
        "last_snapshot": n.snapshots[-1] if len(n.snapshots) else None,
        "buses": len(n.buses),
        "lines": len(n.lines),
        "generators": len(n.generators),
        "loads": len(n.loads),
        "stores": len(n.stores),
        "storage_units": len(n.storage_units),
    }
)
summary.to_frame("value")

## Capacity Mix

This shows optimized installed generator capacity by carrier. When optimized capacity is unavailable for fixed generators, the notebook falls back to nominal capacity.

In [ ]:
def nominal_capacity(df, nominal_col="p_nom", optimized_col="p_nom_opt"):
    if df.empty:
        return pd.Series(dtype=float)
    if optimized_col in df.columns:
        optimized = df[optimized_col].fillna(0.0)
        nominal = df[nominal_col].fillna(0.0) if nominal_col in df.columns else 0.0
        return optimized.where(optimized.abs() > 1e-9, nominal)
    return df[nominal_col].fillna(0.0)

gen_capacity = nominal_capacity(n.generators).groupby(n.generators.carrier).sum().sort_values(ascending=False)
gen_capacity_gw = gen_capacity / 1e3

capacity_df = gen_capacity_gw.rename("capacity_GW").reset_index().rename(columns={"carrier": "carrier"})
display(capacity_df)

fig = px.bar(
    capacity_df,
    x="carrier",
    y="capacity_GW",
    color="carrier",
    title="Generator Capacity by Carrier",
    labels={"capacity_GW": "Capacity (GW)", "carrier": "Carrier"},
)
fig.update_layout(showlegend=False, xaxis_tickangle=-35)
fig.show()

## Energy Mix

Energy is weighted by the model snapshot weighting, so the totals represent the modeled period rather than just raw summed time steps. The pie chart shows each carrier's share of modeled electricity generation.

In [ ]:
snapshot_weight = n.snapshot_weightings.generators.reindex(n.snapshots).fillna(1.0)

if n.generators_t.p.empty:
    generation_gwh = pd.Series(dtype=float)
else:
    generation_mwh_by_generator = n.generators_t.p.multiply(snapshot_weight, axis=0).sum()
    generation_gwh = (
        generation_mwh_by_generator.groupby(n.generators.carrier).sum().sort_values(ascending=False) / 1e3
    )

energy_mix_df = generation_gwh.rename("generation_GWh").reset_index().rename(columns={"carrier": "carrier"})
display(energy_mix_df)

fig = px.pie(
    energy_mix_df,
    names="carrier",
    values="generation_GWh",
    title="Energy Mix by Carrier",
    hole=0.35,
)
fig.update_traces(textposition="inside", textinfo="percent+label")
fig.show()

## Demand And Dispatch

This compares total load with aggregated generator dispatch by carrier. It is useful for spotting whether the solved network is serving demand and which carriers dominate each period.

In [ ]:
load = n.loads_t.p_set.sum(axis=1) if not n.loads_t.p_set.empty else pd.Series(index=n.snapshots, dtype=float)

dispatch_by_carrier = pd.DataFrame(index=n.snapshots)
if not n.generators_t.p.empty:
    dispatch_by_carrier = n.generators_t.p.groupby(n.generators.carrier, axis=1).sum()

dispatch_plot = dispatch_by_carrier.copy()
dispatch_plot["load"] = load
dispatch_long = dispatch_plot.reset_index(names="snapshot").melt(
    id_vars="snapshot", var_name="carrier", value_name="MW"
)

fig = px.area(
    dispatch_long[dispatch_long.carrier != "load"],
    x="snapshot",
    y="MW",
    color="carrier",
    title="Dispatch by Carrier and Total Load",
)
fig.add_trace(
    go.Scatter(
        x=load.index,
        y=load.values,
        mode="lines",
        name="load",
        line=dict(color="black", width=3),
    )
)
fig.update_layout(yaxis_title="MW", xaxis_title="Snapshot")
fig.show()

## Renewable Curtailment

This estimates available renewable output from `p_max_pu * p_nom_opt` and compares it with dispatched output. It is a useful early signal of overbuild, congestion, or insufficient flexibility.

In [ ]:
renewable_carriers = {"solar", "onwind", "offwind-ac", "offwind-dc", "ror", "hydro"}
renewable_gens = n.generators.index[n.generators.carrier.isin(renewable_carriers)]

curtailment = []
for gen in renewable_gens:
    carrier = n.generators.at[gen, "carrier"]
    p_nom = nominal_capacity(n.generators.loc[[gen]]).iloc[0]
    if gen not in n.generators_t.p_max_pu.columns or gen not in n.generators_t.p.columns:
        continue
    available = (n.generators_t.p_max_pu[gen] * p_nom * snapshot_weight).sum()
    dispatched = (n.generators_t.p[gen] * snapshot_weight).sum()
    curtailment.append((carrier, available, dispatched, max(available - dispatched, 0.0)))

curtailment_df = pd.DataFrame(curtailment, columns=["carrier", "available_MWh", "dispatched_MWh", "curtailed_MWh"])
if not curtailment_df.empty:
    curtailment_summary = curtailment_df.groupby("carrier").sum() / 1e3
    curtailment_summary["curtailment_rate_pct"] = (
        curtailment_summary["curtailed_MWh"] / curtailment_summary["available_MWh"].replace(0, pd.NA) * 100
    )
else:
    curtailment_summary = pd.DataFrame(columns=["available_MWh", "dispatched_MWh", "curtailed_MWh", "curtailment_rate_pct"])

display(curtailment_summary.rename(columns={
    "available_MWh": "available_GWh",
    "dispatched_MWh": "dispatched_GWh",
    "curtailed_MWh": "curtailed_GWh",
}))

## Spatial Overview

A quick geographic check: buses and transmission lines should roughly cover Taiwan, and capacity bubbles should sit on modeled buses.

In [ ]:
bus_capacity = n.generators.assign(capacity=nominal_capacity(n.generators)).groupby("bus")["capacity"].sum()
plot_buses = n.buses.join(bus_capacity.rename("generator_capacity_MW")).fillna({"generator_capacity_MW": 0.0})
plot_buses = plot_buses.reset_index(names="bus")
plot_buses["bubble_size"] = 8 + plot_buses["generator_capacity_MW"].clip(lower=0) / max(plot_buses["generator_capacity_MW"].max(), 1) * 42

fig = go.Figure()

for _, line in n.lines.iterrows():
    if line.bus0 in n.buses.index and line.bus1 in n.buses.index:
        b0 = n.buses.loc[line.bus0]
        b1 = n.buses.loc[line.bus1]
        fig.add_trace(
            go.Scattermapbox(
                lon=[b0.x, b1.x],
                lat=[b0.y, b1.y],
                mode="lines",
                line=dict(width=2, color="rgba(80, 80, 80, 0.45)"),
                hoverinfo="skip",
                showlegend=False,
            )
        )

fig.add_trace(
    go.Scattermapbox(
        lon=plot_buses.x,
        lat=plot_buses.y,
        mode="markers+text",
        text=plot_buses.bus,
        textposition="top center",
        marker=dict(
            size=plot_buses.bubble_size,
            color=plot_buses.generator_capacity_MW,
            colorscale="Viridis",
            colorbar=dict(title="MW"),
            opacity=0.82,
        ),
        customdata=plot_buses[["generator_capacity_MW"]],
        hovertemplate="Bus: %{text}<br>Generator capacity: %{customdata[0]:,.0f} MW<extra></extra>",
        name="Buses",
    )
)

center = {"lat": float(plot_buses.y.mean()), "lon": float(plot_buses.x.mean())}
fig.update_layout(
    title="Taiwan Network: Buses, Lines, and Generator Capacity",
    mapbox=dict(style="open-street-map", center=center, zoom=6.5),
    margin=dict(l=0, r=0, t=45, b=0),
    height=700,
)
fig.show()

## Sanity Checks

These checks are not a full validation report, but they catch common early issues: missing solved outputs, NaNs, unserved demand hints, and unusual negative generation.

In [ ]:
checks = {}
checks["result_file_exists"] = RESULT_PATH.exists()
checks["objective_is_finite"] = pd.notna(getattr(n, "objective", None))
checks["load_total_GWh"] = (load * snapshot_weight).sum() / 1e3 if not load.empty else 0.0
checks["generation_total_GWh"] = generation_gwh.sum() if not generation_gwh.empty else 0.0
checks["generator_dispatch_has_nan"] = bool(n.generators_t.p.isna().any().any()) if not n.generators_t.p.empty else False
checks["negative_generator_dispatch_MWh"] = float(
    n.generators_t.p.where(n.generators_t.p < -1e-6, 0).sum().sum()
) if not n.generators_t.p.empty else 0.0

pd.Series(checks).to_frame("value")